# Reducing Reliance on Source Style / 降低对来源文风的依赖（第 4 步）

`02`、`03`、`05` 证明：无论 TF-IDF、LSTM 还是 DistilBERT，只用 ISOT 训练的模型都学会了"认出来源"，换一批来源就失效（WELFake AUC 0.62–0.71）。本 notebook 测试三种应对方法：

| 方法 | 做法 | 训练数据 |
|---|---|---|
| **4a** 只保留内容词 | 训练前删掉虚词和文风标记（"said"、星期几、"Mr."、"just" 等） | ISOT |
| **4b** 去偏重加权 | 先训练一个只看文风词的"偏置模型"；正式训练时，降低它能轻松判对的文章的权重 | ISOT |
| **4c** 多来源训练 | ISOT + WELFake 一起训练，**留出一个出版方**（《纽约时报》或 Breitbart），只在它上面测试 | ISOT + WELFake |

所有方法先用 TF-IDF + 逻辑回归试（几分钟一次），唯一有效的 4c 再用 DistilBERT 确认（3 个种子）。代码见 `src/step4.py`。

**评估规则（避免用测试集挑方法）**：每个模型的 C 在其训练数据的验证集上选；方法之间用 **WELFake val** 比较；**WELFake test** 只在最后报告。

`02`, `03` and `05` showed that every ISOT-trained model learns to recognise the source and fails on other outlets. This notebook tests three remedies — content words only (4a), de-biased reweighting (4b) and multi-source training with a held-out outlet (4c) — first with TF-IDF, then confirming the only effective one (4c) with DistilBERT. Methods are compared on WELFake **val**; WELFake **test** is reported once at the end.

## 1. Setup / 环境

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

sys.path.insert(0, str(Path.cwd().parent))

from src.analysis import paired_bootstrap_auc_diff
from src.external import load_welfake_external
from src.step4 import STYLE_MARKERS, STYLE_WORDS, content_only, style_only

RESULTS = Path('../results/step4')
pd.set_option('display.width', 200)

def load_json(name):
    return json.loads((RESULTS / f'{name}.json').read_text())

def load_probs(name, eval_set, folder=RESULTS):
    d = np.load(folder / f'{name}.npz', allow_pickle=True)
    return pd.Series(d[f'prob::{eval_set}'], index=d[f'doc_id::{eval_set}'].astype(str))

welfake = load_welfake_external().set_index('doc_id')

## 2. What "content words" and "style words" mean / 什么是内容词和文风词

文风词 = 虚词（sklearn 停用词表，**去掉了其中混入的实词**，例如 "bill"、"fire"、"interest" 和数字）+ 文风标记（`04` 和 `05` 中权重最大、换数据集后方向翻转的词）。其余的词都是内容词。两种视角正好互补。

Style words = function words (sklearn's stop-word list **minus the content words it contains**, such as "bill", "fire", "interest" and numbers) + style markers found in `04` / `05`. Everything else is a content word; the two views are exact complements.

In [2]:
print(f'{len(STYLE_WORDS)} style words, of which {len(STYLE_MARKERS)} style markers:')
print(sorted(STYLE_MARKERS))
example = pd.Series(['mr smith said on wednesday that the senate bill would cut taxes just here'])
print('\ntext:          ', example[0])
print('content only:  ', content_only(example)[0])
print('style only:    ', style_only(example)[0])

306 style words, of which 52 style markers:
['according', 'actually', 'added', 'apr', 'aug', 'dec', 'dr', 'even', 'feb', 'friday', 'gov', 'here', 'jan', 'jul', 'jun', 'just', 'literally', 'mar', 'monday', 'mr', 'mrs', 'ms', 'nov', 'oct', 'really', 'rep', 'reported', 'reporters', 'reporting', 'said', 'saturday', 'say', 'saying', 'says', 'sen', 'sep', 'sept', 'spokesman', 'spokesperson', 'spokeswoman', 'stated', 'statement', 'statements', 'sunday', 'tell', 'tells', 'thursday', 'told', 'tuesday', 'video', 'watch', 'wednesday']

text:           mr smith said on wednesday that the senate bill would cut taxes just here
content only:   smith senate bill cut taxes
style only:     mr said on wednesday that the would just here


## 3. 4a and 4b: training on ISOT only / 只用 ISOT 训练的方法

In [3]:
names = {'baseline_full_text': 'baseline (full text)',
         '4a_content_only': '4a content words only',
         'style_only': 'style words only (reference)',
         '4b_reweight_gamma1': '4b reweighting, gamma 1',
         '4b_reweight_gamma2': '4b reweighting, gamma 2'}
rows = {}
for name, label in names.items():
    rec = load_json(name)
    for eval_set, m in rec['metrics'].items():
        rows[(label, eval_set)] = {'C': rec['C'], 'accuracy %': m['accuracy %'],
                                   'balanced accuracy %': m['balanced accuracy %'], 'AUC': m['AUC']}
table_ab = pd.DataFrame(rows).T
order = ['ISOT val', 'WELFake val', 'WELFake test']        # val 在 test 之前：val 是选择依据 / val drives the choice
table_ab.unstack(level=1)[['AUC', 'accuracy %']].reindex(columns=order, level=1).reindex(list(names.values()))

AUC                          accuracy %                         
                             ISOT val WELFake val WELFake test   ISOT val WELFake val WELFake test
baseline (full text)            0.998       0.718        0.714       98.6        55.3         56.7
4a content words only           0.997       0.684        0.669       97.8        55.8         56.8
style words only (reference)    0.993       0.649        0.650       96.6        51.4         52.2
4b reweighting, gamma 1         0.998       0.701        0.698       98.6        53.0         54.5
4b reweighting, gamma 2         0.995       0.678        0.679       96.7        53.7         55.4

**选择依据是 WELFake val 的 AUC**（中间一列）。基线是 0.718，4a 和 4b 都更低，所以都没有改善迁移能力。

The selection criterion is WELFake **val** AUC: baseline 0.718; 4a and 4b are both lower, so neither improves transfer.

In [4]:
# 为什么 4a 没用：只看文风词，或者只看内容词，在 ISOT 上都很高 —— 两种视角都带着来源信息
# Why 4a fails: either view alone scores highly on ISOT, so both carry source information
table_ab.xs('ISOT val', level=1)[['accuracy %', 'AUC']]

,accuracy %,AUC
baseline (full text),98.6,0.998
4a content words only,97.8,0.997
style words only (reference),96.6,0.993
"4b reweighting, gamma 1",98.6,0.998
"4b reweighting, gamma 2",96.7,0.995


In [5]:
# 为什么 4b 没用：偏置模型只看文风词就能判对绝大多数训练文章，权重几乎全部集中在少数文章上
# Why 4b fails: the style-only bias model gets almost every training article right,
# so nearly all the weight lands on a few articles
pd.DataFrame({f'gamma {load_json(n)["gamma"]:g}': {
    'bias model accuracy (out-of-fold)': round(load_json(n)['bias_model_oof_accuracy'], 3),
    **{f'weight at {float(q):.0%} quantile': round(v, 3) for q, v in load_json(n)['weight_quantiles'].items()}}
    for n in ['4b_reweight_gamma1', '4b_reweight_gamma2']})

,gamma 1,gamma 2
bias model accuracy (out-of-fold),0.961,0.961
weight at 10% quantile,0.018,0.030
weight at 50% quantile,0.216,0.030
weight at 90% quantile,2.930,1.722
weight at 99% quantile,10.512,22.170


In [6]:
# 最终报告：WELFake test 上与基线的配对 bootstrap AUC 差 / Final report on WELFake test
base = load_probs('baseline_full_text', 'WELFake test')
y = welfake.loc[base.index, 'label']
pd.DataFrame({names[n]: paired_bootstrap_auc_diff(y, load_probs(n, 'WELFake test').loc[base.index], base)
              for n in ['4a_content_only', '4b_reweight_gamma1', '4b_reweight_gamma2']}).T \
  .rename(columns={'AUC difference (A - B)': 'AUC(method) - AUC(baseline), WELFake test'})

,"AUC(method) - AUC(baseline), WELFake test",95% CI
4a content words only,-0.045,"(-0.052, -0.038)"
"4b reweighting, gamma 1",-0.016,"(-0.022, -0.011)"
"4b reweighting, gamma 2",-0.035,"(-0.047, -0.024)"


## 4. 4c: multi-source training with a held-out outlet / 多来源训练，留出一个出版方

训练：ISOT + WELFake，**去掉被留出出版方的全部文章**。测试：WELFake test 中该出版方的真新闻 + 全部假新闻。分别统计"留出的真新闻被判为真"和"假新闻被误判为真"。

**局限**：测试中的假新闻来自训练时见过的网站（WELFake 没有提供假新闻网站的名称，无法按网站留出），所以分数提高可能有一部分来自"它不像我认识的假新闻网站"。

Training: ISOT + WELFake without any article from the held-out outlet; test: that outlet's real WELFake-test articles + all fake ones. **Limitation**: test fakes come from sites seen in training.

In [7]:
def split_rates(probs):
    y = welfake.loc[probs.index, 'label']
    return {'held-out real -> real %': round(100 * (probs[y == 1] > 0.5).mean(), 1),
            'fakes -> real %': round(100 * (probs[y == 0] > 0.5).mean(), 1),
            'AUC': round(roc_auc_score(y, probs), 3)}

rows = {}
for tag, outlet in [('nyt', 'The New York Times'), ('breitbart', 'Breitbart')]:
    key = f'held-out {outlet} + fakes'
    for m, label in [('isot_only', 'ISOT only'), ('welfake_only', 'WELFake only'),
                     ('isot_plus_welfake', 'ISOT + WELFake'),
                     ('isot_plus_welfake_content_only', 'ISOT + WELFake, content words only')]:
        rows[(outlet, label)] = split_rates(load_probs(f'4c_{tag}_{m}', key))
pd.DataFrame(rows).T

held-out real -> real %  fakes -> real %    AUC
The New York Times ISOT only                                              43.2              9.2  0.790
                   WELFake only                                           88.9              5.2  0.978
                   ISOT + WELFake                                         84.5              5.1  0.968
                   ISOT + WELFake, content words only                     82.6              6.0  0.960
Breitbart          ISOT only                                              10.5              9.2  0.605
                   WELFake only                                           72.3              5.6  0.949
                   ISOT + WELFake                                         43.0              5.0  0.869
                   ISOT + WELFake, content words only                     49.3              6.3  0.865

- 见过多种来源后，模型对 **没见过的** 出版方明显更能接受：《纽约时报》从 43% 提高到 85–89%，Breitbart 从 11% 提高到 43–72%；假新闻误判率也下降了。
- **加入 ISOT 反而有害**，在 Breitbart 上最明显（只用 WELFake 72% → 加入 ISOT 后 43%）：ISOT 教给模型的是"像路透社 = 真"。
- 只保留内容词仍然没有帮助。

Seeing many sources makes an unseen outlet far more acceptable (NYT 43% → 85–89%, Breitbart 11% → 43–72%); adding ISOT hurts, most for Breitbart (72% → 43%); content-only input still does not help.

## 5. Confirming 4c with DistilBERT / 用 DistilBERT 确认 4c

DistilBERT 和 notebook 03 的设置相同，只是训练数据换成"ISOT + WELFake（去掉《纽约时报》）"，3 个种子。对照组是第 2 步只用 ISOT 训练的 DistilBERT（`results/experiments/`），在完全相同的测试文章上比较。

Same DistilBERT settings as notebook 03, trained on ISOT + WELFake without NYT (3 seeds), compared on identical test articles with step 2's ISOT-only DistilBERT.

In [8]:
key = 'held-out The New York Times + fakes'
test_ids = load_probs('4c_nyt_isot_plus_welfake', key).index
step2 = Path('../results/experiments')
rows = {('TF-IDF', 'ISOT only'): split_rates(load_probs('4c_nyt_isot_only', key)),
        ('TF-IDF', 'ISOT + WELFake (4c)'): split_rates(load_probs('4c_nyt_isot_plus_welfake', key))}
for seed in (42, 43, 44):
    only = load_probs(f'distilbert__DistilBERT__seed{seed}', 'WELFake test', folder=step2).loc[test_ids]
    rows[('DistilBERT', f'ISOT only, seed {seed}')] = split_rates(only)
    rows[('DistilBERT', f'ISOT + WELFake (4c), seed {seed}')] = split_rates(load_probs(f'distilbert_4c_seed{seed}', key))
pd.DataFrame(rows).T

held-out real -> real %  fakes -> real %    AUC
TF-IDF     ISOT only                                        43.2              9.2  0.790
           ISOT + WELFake (4c)                              84.5              5.1  0.968
DistilBERT ISOT only, seed 42                               47.2             13.4  0.776
           ISOT + WELFake (4c), seed 42                     77.2              2.1  0.975
           ISOT only, seed 43                               46.8             13.9  0.763
           ISOT + WELFake (4c), seed 43                     82.1              3.2  0.975
           ISOT only, seed 44                               42.7              9.8  0.762
           ISOT + WELFake (4c), seed 44                     74.4              1.9  0.975

In [9]:
# 配对 bootstrap：DistilBERT 4c 与 (a) 只用 ISOT 的 DistilBERT、(b) TF-IDF 4c 比较
# Paired bootstrap: DistilBERT 4c vs (a) ISOT-only DistilBERT and (b) TF-IDF 4c
y = welfake.loc[test_ids, 'label']
tfidf_4c = load_probs('4c_nyt_isot_plus_welfake', key)
rows = {}
for seed in (42, 43, 44):
    bert_4c = load_probs(f'distilbert_4c_seed{seed}', key).loc[test_ids]
    bert_isot = load_probs(f'distilbert__DistilBERT__seed{seed}', 'WELFake test', folder=step2).loc[test_ids]
    for label, other in [('vs ISOT-only DistilBERT', bert_isot), ('vs TF-IDF 4c', tfidf_4c.loc[test_ids])]:
        r = paired_bootstrap_auc_diff(y, bert_4c, other)
        rows[(f'seed {seed}', label)] = {'AUC(DistilBERT 4c) - AUC(other)': r['AUC difference (A - B)'],
                                         '95% CI': r['95% CI']}
pd.DataFrame(rows).T

AUC(DistilBERT 4c) - AUC(other)          95% CI
seed 42 vs ISOT-only DistilBERT                           0.199  (0.185, 0.214)
        vs TF-IDF 4c                                      0.007  (0.002, 0.013)
seed 43 vs ISOT-only DistilBERT                           0.211  (0.196, 0.226)
        vs TF-IDF 4c                                      0.006  (0.001, 0.013)
seed 44 vs ISOT-only DistilBERT                           0.212  (0.197, 0.228)
        vs TF-IDF 4c                                      0.007  (0.001, 0.013)

## 6. Conclusions / 结论

**1. 只用 ISOT 训练时，删词和重加权都没有用，而且都让迁移变差了。**

| 方法 | WELFake val AUC（选择依据） | WELFake test 与基线的 AUC 差（95% 置信区间） |
|---|---|---|
| 基线（完整文本） | 0.718 | — |
| 4a 只保留内容词 | 0.684 | −0.045（−0.052 到 −0.038） |
| 4b 重加权 gamma 1 | 0.701 | −0.016（−0.022 到 −0.011） |
| 4b 重加权 gamma 2 | 0.678 | −0.035（−0.047 到 −0.024） |

三种做法在 WELFake test 上都 **可靠地变差**（置信区间都不包含 0）。

**2. 原因：来源信息同时藏在文风和话题里。**
- 只看文风词，ISOT 上就有 **96.6%**；只看内容词，也有 **97.8%**。删掉其中一种，另一种照样能暴露来源（例如路透社多写国际新闻和部长，假新闻网站多写 Hillary 和 Obama）。
- 4b 的偏置模型只看文风词，就能判对 **96.1%** 的训练文章，所以权重几乎全部集中在剩下的少数文章上（gamma 1 时，权重中位数只有 0.22，最高的 1% 文章超过 10）。训练实际上只用了很少的文章，学不出更好的模型。

**3. 唯一有效的是改变训练数据（4c）。**
见过多种来源后，模型对 **没见过的** 出版方明显更能接受，同时假新闻的误判也减少了：

| 留出的出版方 | 训练数据（TF-IDF） | 留出的真新闻被判为真 | 假新闻被误判为真 | AUC |
|---|---|---|---|---|
| 《纽约时报》 | 只用 ISOT | 43.2% | 9.2% | 0.790 |
| | ISOT + WELFake | 84.5% | 5.1% | 0.968 |
| | 只用 WELFake | 88.9% | 5.2% | 0.978 |
| Breitbart | 只用 ISOT | 10.5% | 9.2% | 0.605 |
| | ISOT + WELFake | 43.0% | 5.0% | 0.869 |
| | 只用 WELFake | 72.3% | 5.6% | 0.949 |

**加入 ISOT 反而有害**，在 Breitbart 上最明显（72.3% → 43.0%）：ISOT 教给模型的是"像路透社 = 真"。

**4. DistilBERT 确认了 4c 的效果，而且它带来的提升远大于换模型带来的提升。**
- DistilBERT 改用 ISOT + WELFake 训练后，在留出的《纽约时报》测试集上，AUC 从 0.762–0.776 提高到 **0.975**（3 个种子的提升都在 +0.199 到 +0.212 之间，置信区间都不包含 0）。《纽约时报》被接受的比例从 43–47% 提高到 74–82%，假新闻误判率从 10–14% 降到 2–3%。
- 同样用 4c 训练，DistilBERT 比 TF-IDF 只高 0.006–0.007（可靠，但很小）。
- **改变训练数据带来约 +0.2 的 AUC，换更强的模型只带来约 +0.007。** 瓶颈在数据，不在模型。

**局限**：
- 测试中的假新闻来自训练时见过的网站，所以 4c 的提升可能有一部分是"它不像我认识的假新闻网站"，而不全是"学会了接受新的真新闻来源"。
- 标签仍然是按来源打的，所以 4c 说明的是"能更好地应对没见过的来源"，**不能说明模型学会了判断真假**。后者要在逐条核查的数据（LIAR，见 `07_liar_benchmark`）上检验。

---

1. **Training on ISOT only, removing style words (4a) or reweighting (4b) makes transfer reliably worse** on WELFake test (−0.045, −0.016, −0.035 AUC; all CIs exclude 0).
2. **Source information is in both style and topic**: style words alone reach 96.6% on ISOT, content words alone 97.8%. The style-only bias model gets 96.1% of training articles right, so 4b concentrates nearly all weight on a few articles.
3. **Only changing the training data helps (4c)**: an unseen NYT is accepted as real 43% → 85–89% of the time; Breitbart 11% → 43–72%; adding ISOT hurts (Breitbart 72% → 43%).
4. **DistilBERT confirms 4c** (AUC 0.762–0.776 → 0.975, +0.20 to +0.21), while DistilBERT vs TF-IDF under 4c differs by only +0.007: **data matters far more than the model.**

**Limitations**: test fakes come from sites seen in training, and labels are still per source, so 4c shows better handling of unseen sources, not truthfulness detection — that is tested on LIAR (`07_liar_benchmark`).